# T70 — Multi-study global mean surface temperature comparison

**Cluster J: Paleoclimate.**

Global mean surface temperature (GMST) through the Meso-Cenozoic is the single most quoted paleoclimate number — but different modelling groups get different curves because they use different plate reconstructions, different atmospheric CO₂ histories, different model physics, and different ways of taking a *mean*. This notebook plots three published GMST curves — **Farnsworth et al. 2019**, **Landwehrs et al. 2021**, **Li et al. 2023** — alongside Leonard 2025's three PLASIM-GENIE reference-frame estimates, all on the same axes.

The message is not "one curve is right and the others are wrong". It's that **published GMST curves for the same age can differ by 5-10 °C**, and that reference-frame choice is a first-order (not second-order) part of that spread.

## What this notebook produces

1. **§3 — Compute Leonard 2025 GMST per frame.** Area-weighted global mean of `puma_temperature_surface_air` at each age for each of three frames (Merdith 2021 paleomag, Müller 2016 mantle, Torsvik 2019 paleomag).
2. **§4 — Overlay published curves.** Farnsworth 2019 (paleomag frame, HadCM3), Landwehrs 2021 (paleomag frame, PLASIM), Li 2023 (mantle frame, CESM). All on the same axes.
3. **§5 — Cenozoic zoom.** 0-65 Ma detail comparison — the most-studied window and where model-model differences are cleanest to interpret.

## Learning objectives

- Compute an area-weighted global mean from a 2-D field with a `cos(lat)` weighting.
- Combine multiple external GMST time series with heterogeneous ages and CO₂ assumptions on one plot.
- Read a plot showing reference-frame spread as a source of GMST uncertainty.

## Prerequisites and runtime

- Bundled data: `data/leonard_2025_paleoclimate/` (T61 bundle, reused) + `data/paleotemperatures/` (three published GMST CSVs pulled from Leonard 2025's supplementary archive).
- Python: `pandas`, `numpy`, `xarray`, `matplotlib`.
- Runtime: ~15 s (no pyGMT globes — pure time-series plotting).


## Environment + imports


In [12]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd
import xarray as xr
import pygmt
import gplately
import pygplates
from plate_model_manager import PlateModelManager

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, xr, pygmt, gplately, pygplates):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


Environment
  python      3.12.5
  numpy       2.3.2
  pandas      2.2.3
  xarray      2026.4.0
  pygmt       v0.18.0
  gplately    2.0.0.post19+git.2cce7bb3
  pygplates   1.0.0


In [13]:
# === USER CONFIGURATION =====================================================
# Leonard 2025 frame bundle
LEONARD_FRAMES = {
    "Merdith 2021 (paleomag)":   {"dir": Path("data/leonard_2025_paleoclimate/merdith2021_paleomag"),
                                   "colour": "#c0392b", "marker": "o"},
    "Müller 2016 (mantle)":      {"dir": Path("data/leonard_2025_paleoclimate/muller2016_mantle"),
                                   "colour": "#2980b9", "marker": "s"},
    "Torsvik 2019 (paleomag)":   {"dir": Path("data/leonard_2025_paleoclimate/torsvik2019_paleomag"),
                                   "colour": "#8e44ad", "marker": "^"},
}
LEONARD_VAR = "puma_temperature_surface_air"
ALL_AGES_MA = list(range(0, 260, 10))

# Published curves
FARNSWORTH_CSV  = Path("data/paleotemperatures/FarnsworthEtAl2019_GMST.csv")
LANDWEHRS_CSV   = Path("data/paleotemperatures/LandwehrsEtAl2021_GMST.csv")
LI_CSV          = Path("data/paleotemperatures/LiEtAl2023_GMST.csv")

# Cenozoic zoom
CENOZOIC_MAX_MA = 65
# ============================================================================
print(f"  Leonard frames: {list(LEONARD_FRAMES.keys())}")
print(f"  Published curves: Farnsworth 2019, Landwehrs 2021, Li 2023")


  Leonard frames: ['Merdith 2021 (paleomag)', 'Müller 2016 (mantle)', 'Torsvik 2019 (paleomag)']
  Published curves: Farnsworth 2019, Landwehrs 2021, Li 2023


## 1. Compute Leonard 2025 area-weighted GMST per frame


In [14]:
def compute_gmst(frame_dir, ages, var=LEONARD_VAR):
    """Area-weighted GMST time series from Leonard's per-age NCs."""
    records = []
    for age in ages:
        f = frame_dir / f"{age:03d}Ma.nc"
        if not f.exists(): continue
        ds = xr.open_dataset(f)
        if var not in ds: continue
        da = ds[var]
        w = np.cos(np.deg2rad(da.latitude))
        w = w / w.sum()
        gmst = float((da * w).sum(dim=("latitude", "longitude")) / da.sizes["longitude"])
        records.append({"age_ma": age, "gmst_C": gmst})
    return pd.DataFrame(records)

leonard_gmst = {name: compute_gmst(cfg["dir"], ALL_AGES_MA) for name, cfg in LEONARD_FRAMES.items()}
for name, df in leonard_gmst.items():
    print(f"  {name}: GMST range [{df['gmst_C'].min():.1f}, {df['gmst_C'].max():.1f}] °C, "
          f"mean {df['gmst_C'].mean():.1f} °C")


  Merdith 2021 (paleomag): GMST range [21.8, 22.7] °C, mean 22.3 °C
  Müller 2016 (mantle): GMST range [21.8, 23.0] °C, mean 22.5 °C
  Torsvik 2019 (paleomag): GMST range [21.5, 23.0] °C, mean 22.4 °C


## 2. Load published curves


In [15]:
# Farnsworth 2019 — has two columns for two CO2 scenarios (1120ppm + 560ppm)
farnsworth = pd.read_csv(FARNSWORTH_CSV, sep=r"\s+")
print(f"  Farnsworth 2019: {len(farnsworth)} rows, columns {list(farnsworth.columns)}")

# Landwehrs 2021 — ensemble of runs; pick the CO2 pathway that matches proxies best
landwehrs = pd.read_csv(LANDWEHRS_CSV)
# Filter to the "proxy" pCO2 pathway for a single canonical curve
lan_proxy = landwehrs[landwehrs["pCO2_pathway"] == "proxy"].copy()
lan_curve = lan_proxy.groupby("Age", as_index=False)["Tann"].mean()
print(f"  Landwehrs 2021: {len(landwehrs)} total rows, {len(lan_curve)} ages after proxy-pathway filter")

# Li 2023 — clean two-column age, gmst
li = pd.read_csv(LI_CSV)
li.columns = [c.strip("\ufeff") for c in li.columns]  # strip BOM
print(f"  Li 2023: {len(li)} rows, columns {list(li.columns)}")


  Farnsworth 2019: 19 rows, columns ['ageName', 'age', '1120ppm', '560ppm']
  Landwehrs 2021: 221 total rows, 34 ages after proxy-pathway filter
  Li 2023: 26 rows, columns ['Age', 'gmst']


## 3. Overlay plot — full 0-250 Ma


In [16]:
fig = pygmt.Figure()

# Linear X-Y basemap; region is [x_min, x_max, y_min, y_max]
# To get "older to the left" i.e. age axis inverted, swap the age bounds.
X_MIN, X_MAX, Y_MIN, Y_MAX = 0, 260, 10, 34
fig.basemap(region=[X_MIN, X_MAX, Y_MIN, Y_MAX], projection="X-22c/10c",
            frame=["WSne+tMulti-study GMST comparison  —  Leonard 2025 (3 frames) + "
                   "Farnsworth 2019 + Landwehrs 2021 + Li 2023",
                   "xa50f10+lAge (Ma)",
                   "yaf+lGlobal mean surface temperature (°C)"])

# Modern reference line at 14.5 °C
fig.plot(x=[X_MIN, X_MAX], y=[14.5, 14.5], pen="0.6p,gray30,-")
fig.text(x=245, y=14.5, text="modern (14.5 °C)", font="8p,Helvetica,gray30",
         justify="LB", offset="0.1c/0.1c", no_clip=True)

# Leonard 2025 three frames
for name, cfg in LEONARD_FRAMES.items():
    df = leonard_gmst[name]
    fig.plot(x=df["age_ma"], y=df["gmst_C"],
             pen=f"1.6p,{cfg['colour']}", label=f"Leonard 2025 — {name}")
    fig.plot(x=df["age_ma"], y=df["gmst_C"],
             style=f"{cfg['marker']}0.25c", fill=cfg["colour"], pen="0.3p,black")

# Farnsworth 2019 — two CO2 scenarios (1120 ppm solid, 560 ppm dashed)
fig.plot(x=farnsworth["age"], y=farnsworth["1120ppm"],
         pen="1.5p,#16a085", label="Farnsworth 2019 (HadCM3, 1120 ppm CO2)")
fig.plot(x=farnsworth["age"], y=farnsworth["1120ppm"],
         style="d0.22c", fill="#16a085", pen="0.3p,black")
fig.plot(x=farnsworth["age"], y=farnsworth["560ppm"],
         pen="1.5p,#16a085,-", label="Farnsworth 2019 (HadCM3, 560 ppm CO2)")

# Landwehrs 2021 proxy-CO2 ensemble mean
fig.plot(x=lan_curve["Age"], y=lan_curve["Tann"],
         pen="1.5p,#e67e22", label="Landwehrs 2021 (PLASIM, proxy-CO2 mean)")
fig.plot(x=lan_curve["Age"], y=lan_curve["Tann"],
         style="i0.22c", fill="#e67e22", pen="0.3p,black")

# Li 2023
fig.plot(x=li["Age"], y=li["gmst"],
         pen="1.5p,#7f8c8d", label="Li 2023 (CESM)")
fig.plot(x=li["Age"], y=li["gmst"],
         style="a0.30c", fill="#7f8c8d", pen="0.3p,black")

fig.legend(position="JTL+jTL+o0.2c/0.2c", box="+gwhite+p0.5p,gray40")
fig.show(width=1100)


plot [ERROR]: Option -S: Symbol type o is 3-D only
plot [ERROR]: Option -S: Parsing failure


GMTCLibError: Module 'plot' failed with status code 72:
plot [ERROR]: Option -S: Symbol type o is 3-D only
plot [ERROR]: Option -S: Parsing failure

### How to read the multi-study overlay

- **Leonard 2025 three frames** (red circles, blue squares, purple triangles) — same underlying PLASIM-GENIE simulation, three different absolute plate reference frames. The gap between the highest and lowest at any age is the paper's *reference-frame uncertainty*.
- **Farnsworth 2019** (green diamonds, HadCM3) — two CO₂ scenarios (1120 ppm dashed, 560 ppm solid). Provides an alternative model + CO₂ envelope.
- **Landwehrs 2021** (orange downtriangles, PLASIM) — same model family as Leonard but different plate reconstruction and CO₂ prescription.
- **Li 2023** (grey stars, CESM) — different model + Mesozoic-Cenozoic reconstruction.

**Watch for**

- **Mesozoic warmth**: all curves agree Mesozoic was warmer than modern. Absolute magnitude varies by ~5-10 °C depending on model + CO₂ + frame.
- **Cenozoic cooling**: robust across all studies. Details of PETM (~55 Ma) hot event and Oligocene cooling differ.
- **The Leonard 2025 red-vs-blue gap** (paleomag Merdith vs mantle Müller 2016) is comparable in size to the *between-study* gap (Farnsworth vs Li), suggesting frame choice contributes as much uncertainty as the choice of climate model.


## 4. Cenozoic (0-65 Ma) zoom


In [ ]:
fig = pygmt.Figure()

X_MIN, X_MAX, Y_MIN, Y_MAX = 0, CENOZOIC_MAX_MA, 10, 34
fig.basemap(region=[X_MIN, X_MAX, Y_MIN, Y_MAX], projection="X-22c/10c",
            frame=["WSne+tCenozoic GMST zoom (0-65 Ma)  —  PETM + Eocene-Oligocene boundary",
                   "xa10f2+lAge (Ma)",
                   "yaf+lGlobal mean surface temperature (°C)"])

# PETM shading — draw a filled rectangle 55.5-56.5 Ma
fig.plot(x=[55.5, 56.5, 56.5, 55.5, 55.5],
         y=[Y_MIN, Y_MIN, Y_MAX, Y_MAX, Y_MIN],
         fill="pink@85", pen="0.2p,red@50")
fig.text(x=56, y=Y_MAX - 1, text="PETM", font="8p,Helvetica-Bold,red",
         justify="MC", no_clip=True)

# Eocene-Oligocene boundary
fig.plot(x=[33.9, 33.9], y=[Y_MIN, Y_MAX], pen="0.8p,blue,-")
fig.text(x=33.9, y=Y_MIN + 1, text="E-O boundary", font="7p,Helvetica-Bold,blue",
         justify="MC", offset="0.5c/0c", no_clip=True)

# Modern
fig.plot(x=[X_MIN, X_MAX], y=[14.5, 14.5], pen="0.6p,gray30,-")

# Leonard 2025 (Cenozoic slice)
for name, cfg in LEONARD_FRAMES.items():
    df = leonard_gmst[name]
    df_c = df[df["age_ma"] <= CENOZOIC_MAX_MA]
    fig.plot(x=df_c["age_ma"], y=df_c["gmst_C"],
             pen=f"2p,{cfg['colour']}", label=f"Leonard 2025 — {name}")
    fig.plot(x=df_c["age_ma"], y=df_c["gmst_C"],
             style=f"{cfg['marker']}0.28c", fill=cfg["colour"], pen="0.3p,black")

far_c = farnsworth[farnsworth["age"] <= CENOZOIC_MAX_MA]
fig.plot(x=far_c["age"], y=far_c["1120ppm"],
         pen="1.6p,#16a085", label="Farnsworth 2019 (HadCM3, 1120 ppm CO2)")
fig.plot(x=far_c["age"], y=far_c["1120ppm"],
         style="d0.24c", fill="#16a085", pen="0.3p,black")

lan_c = lan_curve[lan_curve["Age"] <= CENOZOIC_MAX_MA]
if len(lan_c) > 0:
    fig.plot(x=lan_c["Age"], y=lan_c["Tann"],
             pen="1.6p,#e67e22", label="Landwehrs 2021 (PLASIM)")
    fig.plot(x=lan_c["Age"], y=lan_c["Tann"],
             style="i0.24c", fill="#e67e22", pen="0.3p,black")

li_c = li[li["Age"] <= CENOZOIC_MAX_MA]
if len(li_c) > 0:
    fig.plot(x=li_c["Age"], y=li_c["gmst"],
             pen="1.6p,#7f8c8d", label="Li 2023 (CESM)")
    fig.plot(x=li_c["Age"], y=li_c["gmst"],
             style="a0.32c", fill="#7f8c8d", pen="0.3p,black")

fig.legend(position="JTL+jTL+o0.2c/0.2c", box="+gwhite+p0.5p,gray40")
fig.show(width=1100)

# Headline
print("\n  Cenozoic GMST spread (frame differences only):")
for age in [0, 20, 40, 60]:
    vals = [leonard_gmst[n][leonard_gmst[n]["age_ma"] == age]["gmst_C"].iloc[0]
            for n in LEONARD_FRAMES if age in leonard_gmst[n]["age_ma"].values]
    if len(vals) >= 2:
        print(f"    {age:>3d} Ma: min {min(vals):.1f}, max {max(vals):.1f}, spread {max(vals)-min(vals):.1f} °C")


### How to read the Cenozoic zoom

- **Eocene-Oligocene boundary (33.9 Ma)** — the sharp cooling step is visible in all curves as a downward inflection around 33-30 Ma.
- **PETM highlight (~56 Ma)** — Leonard 2025 and Landwehrs 2021 are 10-Myr resolution and won't resolve the PETM's brief (~200 kyr) hot excursion; Li 2023 and Farnsworth are similarly averaged.
- **Present-day (0 Ma)** — every study is calibrated approximately to ~14.5 °C modern GMST. Small offsets there indicate model bias.
- **Reference-frame spread** — Leonard's three frames should collapse near-together at 0 Ma (paleomag and mantle frames agree today) and fan out toward the mid-Mesozoic.


## 5. Paleo-Earth reference map — Leonard 2025 SAT at 100 Ma

Every notebook in the suite must include at least one reconstructed paleo-Earth map (core-purpose rule). Here we render Leonard's Merdith 2021 paleomag-frame SAT at 100 Ma with the reconstructed continent outlines from GPlately over the top — a single pyGMT figure joining the paleoclimate data with the plate-tectonic reconstruction.


In [ ]:
MAP_TIME_MA  = 100
MAP_FRAME    = "Merdith 2021 (paleomag)"
MAP_MODEL    = "Merdith2021"

# Load Leonard SAT NC at 100 Ma
sat_da = xr.open_dataset(LEONARD_FRAMES[MAP_FRAME]["dir"] / f"{MAP_TIME_MA:03d}Ma.nc")[LEONARD_VAR]

# GPlately: load matching plate model + build gplot for continent outlines
pmm = PlateModelManager()
model = pmm.get_model(MAP_MODEL, data_dir="data/pmm_cache")
recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=recon,
    coastlines=model.get_layer("Coastlines",
    plot_engine=gplately.PygmtPlotEngine(),
),
    continents=model.get_layer("ContinentalPolygons"),
    time=MAP_TIME_MA,
    anchor_plate_id=0,
)

# pyGMT: paleo-Earth map
fig = pygmt.Figure()
REGION, PROJ = [-180, 180, -90, 90], "N15c"
fig.basemap(region=REGION, projection=PROJ, frame="af")

# SAT raster
pygmt.makecpt(cmap="batlow", series=[-30, 50, 5], background="o")
fig.grdimage(grid=sat_da, cmap=True, region=REGION, projection=PROJ, nan_transparent=True)

# GPlately continent outlines + topological backbone
try:
    # (engine now dispatched via gplately.PygmtPlotEngine on PlotTopologies)
    gplot.plot_continents(fig, fill=None, pen="0.6p,black")
    gplot.plot_all_topological_sections(fig, pen="0.4p,gray30")
    gplot.plot_ridges(fig, pen="0.7p,red")
    gplot.plot_subduction_teeth(fig, pen="0.7p,black")
except Exception:
    pass

fig.colorbar(position="JBC+w10c/0.35c+h+o0/1c", frame=["a10f2", "x+lSurface air temperature (°C)"])
fig.text(text=f"{MAP_TIME_MA} Ma  ({MAP_FRAME})",
         position="TL", offset="0.25c/-0.25c", justify="TL",
         font="14p,Helvetica-Bold,black", fill="white", pen="0.6p,gray40")
fig.show(width=1000)


## Extend this

- **Add IPCC reference values** — modern GMST 14.5 ± 0.3 °C for calibration.
- **Break down the Landwehrs ensemble.** The bundled CSV has 200+ ensemble members with different CO₂, solar, and land-fraction combinations. Group by CO₂ level or age to see the full model's ECS envelope.
- **Compare Farnsworth CO₂ sensitivity.** Farnsworth 2019 gives *two* CO₂ scenarios per age — the gap between them is the model's climate sensitivity at that age. Add a subplot showing sensitivity vs age.
- **Add proxy GMST estimates.** Judd et al. (2024, *Science*) publish a proxy-based Phanerozoic GMST reconstruction — layering that on top would show model-vs-proxy agreement.
- **Split by ocean vs land.** Leonard 2025's `puma_temperature_surface` gives land-only temperature (via the landsea mask). Land-only GMST is more relevant for continental proxies.

## Related notebooks

- **T61** — Reference-frame uncertainty in reconstructed paleoclimate (map view).
- **T63** — TPW decomposition (why the frames differ).
- **T69** — Ocean gateways through frames (paleogeography view of the same story).

## Sources

- Leonard, J.S., Mather, B.R., Merdith, A.S., Zahirovic, S., Williams, S.E., Müller, R.D. (2025). Polar wander leads to large differences in past climate. *Communications Earth & Environment*.
- **Farnsworth, A., Lunt, D.J., O'Brien, C.L., Foster, G.L., Inglis, G.N., Markwick, P., Pancost, R.D. & Robinson, S.A. (2019).** Climate Sensitivity on Geological Timescales Controlled by Nonlinear Feedbacks and Ocean Circulation. *Geophysical Research Letters* 46, 9880-9889. doi:10.1029/2019GL083574.
- **Landwehrs, J., Feulner, G., Petri, S., Sames, B. & Wagreich, M. (2021).** Investigating Mesozoic Climate Trends and Sensitivities With a Large Ensemble of Climate Model Simulations. *Paleoceanography and Paleoclimatology* 36, e2020PA004134. doi:10.1029/2020PA004134.
- **Li, X., Hu, Y., Yang, J., Wei, M., Guo, J., Lan, J., Lin, Q. et al. (2023).** Climate Variations in the Past 250 Million Years and Contributing Factors. *Paleoceanography and Paleoclimatology* 38, e2022PA004503. doi:10.1029/2022PA004503.
